# ESM-DMS real-data analysis

This notebook runs the real-data ESM-DMS workflow through the current `esmDMS` class structure for cellular datasets `TpoR`, `Ube4b`, and `BRCA1`, and viral datasets `BF520` and `BG505`.

Notebook code is limited to configuration and orchestration around `esmDMS` methods. Missing class-level affordances are marked with `# TODO(esmDMS.py)`.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from esmDMS import CellularDMSInput, ViralDMSInput, ESMDMSConfig, esmDMS

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break

DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw_data"
SEQUENCE_DIR = DATA_DIR / "sequence_data"
ANALYSIS_DIR = DATA_DIR / "esm_data_analysis"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR = ANALYSIS_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="darkgrid")
REPO_ROOT


## Dataset Registry

Cellular datasets use MaveDB nucleotide-count files. Viral datasets use paired mutant-DNA and mutant-virus codon-count files.


In [ ]:
CELLULAR_DATASETS = {
    "TpoR": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "TpoR_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "TpoR_nucleotide_counts.csv",
    ),
    "Ube4b": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "Ube4b_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "Ube4b_nucleotide_counts.csv",
    ),
    "BRCA1": CellularDMSInput(
        reference_nuc_path=RAW_DIR / "BRCA1_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "BRCA1_nucleotide_counts.csv",
    ),
}

VIRAL_DATASETS = {
    "BF520": ViralDMSInput(
        pre_files=tuple(RAW_DIR / f"BF520_mutDNA-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
        post_files=tuple(RAW_DIR / f"BF520_mutvirus-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
    ),
    "BG505": ViralDMSInput(
        pre_files=tuple(RAW_DIR / f"BG505_mutDNA-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
        post_files=tuple(RAW_DIR / f"BG505_mutvirus-{rep}_codoncounts.csv" for rep in (1, 2, 3)),
    ),
}

DATASETS = {**CELLULAR_DATASETS, **VIRAL_DATASETS}
DATASET_KIND = {
    **{dataset: "cellular" for dataset in CELLULAR_DATASETS},
    **{dataset: "viral" for dataset in VIRAL_DATASETS},
}

pd.DataFrame(
    {"dataset": dataset, "kind": DATASET_KIND[dataset], "save_dir": str(SEQUENCE_DIR / dataset)}
    for dataset in DATASETS
)


## Controls

`RUN_EMBEDDING` is off by default because ESM embedding is the expensive step. Turn it on only when the current class-format embedding files are absent or need to be regenerated.


In [ ]:
# TODO(esmDMS.py): add a public available_layers() method so notebooks do not hard-code model layers.
LAYERS = list(range(34))
REPRESENTATIVE_LAYER = 33

ABSTRACTION_METHOD = "Embeddings"
ABSTRACTION_PARAMS = {"norm_scheme": "none"}
NORM_SCHEME = ABSTRACTION_PARAMS["norm_scheme"]

RUN_EMBEDDING = False
RUN_INFERENCE = True


## Create esmDMS Runners

Each dataset gets its own `esmDMS` object with a dataset-specific save directory.


In [ ]:
runners = {}

for dataset, input_data in DATASETS.items():
    config = ESMDMSConfig(
        embedding_model="esm2_t33_650M_UR50D",
        embedding_method="per_residue",
        local_or_disk="disk",
        save_dir=str(SEQUENCE_DIR / dataset),
        dataset_name=dataset,
    )
    runners[dataset] = esmDMS(input_data=input_data, config=config)

runners


## Process Raw Data

Raw input parsing is handled by `esmDMS.process_raw_data()`.


In [ ]:
processing_rows = []

for dataset, runner in runners.items():
    runner.process_raw_data(drop_stop_codons=True)
    df = runner.sequence_dataframe
    processing_rows.append({
        "dataset": dataset,
        "kind": DATASET_KIND[dataset],
        "rows": len(df),
        "sequence_count": df["SequenceIndex"].nunique(),
        "replicate_count": df["Replicate"].nunique(),
        "generation_count": df["Generation"].nunique(),
    })

processing_summary = pd.DataFrame(processing_rows)
processing_summary.to_csv(TABLE_DIR / "raw_processing_summary.csv", index=False)
processing_summary


## Current Class Cache Status

This checks the paths that the current `esmDMS` class will use.


In [ ]:
# TODO(esmDMS.py): add a public cache_status(layers, abstraction_method, norm_scheme) method.
cache_rows = []

for dataset, runner in runners.items():
    for layer in LAYERS:
        cache_rows.append({
            "dataset": dataset,
            "layer": layer,
            "embedding_path": str(runner._embedding_path(layer)),
            "embedding_exists": runner._embedding_path(layer).exists(),
            "inference_path": str(runner._inference_path(ABSTRACTION_METHOD, layer, NORM_SCHEME)),
            "inference_exists": runner._inference_path(ABSTRACTION_METHOD, layer, NORM_SCHEME).exists(),
        })

cache_status = pd.DataFrame(cache_rows)
cache_status.to_csv(TABLE_DIR / "current_class_cache_status.csv", index=False)
cache_status


## Embed Sequences

`esmDMS.embed_all_sequences()` performs ESM embedding and writes class-format embedding caches. This is skipped by default because it is expensive.


In [ ]:
if RUN_EMBEDDING:
    for runner in runners.values():
        runner.embed_all_sequences(layer="all")


## Run Or Load Inference

`esmDMS.run_feature_inference()` loads existing class-format inference results when present, otherwise it loads class-format embeddings and runs inference.


In [ ]:
# TODO(esmDMS.py): add force_recompute to run_feature_inference() if cache invalidation is needed.
inference_results = {}

if RUN_INFERENCE:
    for dataset, runner in runners.items():
        inference_results[dataset] = {}
        for layer in LAYERS:
            inference_results[dataset][layer] = runner.run_feature_inference(
                layer=layer,
                abstraction_method=ABSTRACTION_METHOD,
                abstraction_params=ABSTRACTION_PARAMS,
            )


## Inference Summary

The notebook only reads fields from the `InferenceResult` objects returned by `esmDMS.run_feature_inference()`.


In [ ]:
# TODO(esmDMS.py): add an inference_summary() method that returns this table from stored results.
inference_rows = []

for dataset, layer_results in inference_results.items():
    for layer, result in layer_results.items():
        inference_rows.append({
            "dataset": dataset,
            "kind": DATASET_KIND[dataset],
            "layer": layer,
            "n_replicates": result.s.shape[0],
            "n_dimensions": result.s.shape[1],
            "gamma_opt": result.gamma_opt,
            "s_joint_mean": result.s_joint.mean(),
            "s_joint_std": result.s_joint.std(),
        })

inference_summary = pd.DataFrame(inference_rows)
inference_summary.to_csv(TABLE_DIR / "inference_result_summary.csv", index=False)
inference_summary


## Replicate Consistency By Layer

Layer-wise replicate consistency plots are delegated to `esmDMS.plot_avg_rep_correlations_by_layer()`.


In [ ]:
for dataset, runner in runners.items():
    runner.plot_avg_rep_correlations_by_layer(
        layers=LAYERS,
        abstraction_method=ABSTRACTION_METHOD,
        norm_scheme=NORM_SCHEME,
        comparison="selection",
        label=dataset,
        output_path=FIGURE_DIR / f"{dataset}_selection_replicate_correlations_by_layer.png",
    )
    runner.plot_avg_rep_correlations_by_layer(
        layers=LAYERS,
        abstraction_method=ABSTRACTION_METHOD,
        norm_scheme=NORM_SCHEME,
        comparison="fitness",
        label=dataset,
        output_path=FIGURE_DIR / f"{dataset}_fitness_replicate_correlations_by_layer.png",
    )
    plt.close("all")


## Representative Replicate Scatter Plots

Replicate scatter plots are delegated to `esmDMS.plot_rep_sel_comps()` and `esmDMS.plot_rep_fit_comps()`.


In [ ]:
for dataset, runner in runners.items():
    runner.plot_rep_sel_comps(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        norm_scheme=NORM_SCHEME,
        label=dataset,
        output_path=FIGURE_DIR / f"{dataset}_layer{REPRESENTATIVE_LAYER}_selection_replicate_scatter.png",
    )
    runner.plot_rep_fit_comps(
        layer=REPRESENTATIVE_LAYER,
        abstraction_method=ABSTRACTION_METHOD,
        norm_scheme=NORM_SCHEME,
        label=dataset,
        output_path=FIGURE_DIR / f"{dataset}_layer{REPRESENTATIVE_LAYER}_fitness_replicate_scatter.png",
    )
    plt.close("all")


## Shuffled-Frequency Control

This notebook does not implement shuffled controls locally.


In [ ]:
# TODO(esmDMS.py): add a class method for shuffled-frequency controls that shuffles
# within each (Replicate, Generation), reruns inference, and returns InferenceResult
# objects compatible with the plotting methods above.
